# CNN Depth-Width Ratio Experiments

This notebook is for running the project experiments on Google Colab. It stores the project code, datasets, and experiment outputs under `MyDrive/5329A2/`, then runs fixed-parameter-budget sweeps for plain and residual CNNs.

Recommended Colab runtime: **Runtime > Change runtime type > T4 GPU**.

## 1. Check GPU

Run this first. If `torch.cuda.is_available()` is `False`, switch the runtime to GPU before starting long experiments.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Use Runtime > Change runtime type > GPU.")

## 2. Mount Google Drive And Set Persistent Paths

All project-related files are stored in `MyDrive/5329A2/` so Colab runtime resets do not delete the code, CIFAR data, or experiment outputs.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/5329A2')
PROJECT_DIR = DRIVE_ROOT / 'project'
DATA_DIR = DRIVE_ROOT / 'data'
RESULTS_DIR = DRIVE_ROOT / 'results'

for path in [DRIVE_ROOT, DATA_DIR, RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)
print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

## 3. Prepare Project Code In Drive And Install Dependencies

The repository is cloned into `MyDrive/5329A2/project/`. If it already exists, this cell runs `git pull` instead of cloning again. Do not paste personal access tokens into the notebook.

In [ ]:
import os

REPO_URL = "https://github.com/kwokxxx/Cnn-depth-width-ratio-study.git"

if (PROJECT_DIR / '.git').exists():
    %cd {PROJECT_DIR}
    !git pull --ff-only
else:
    %cd {DRIVE_ROOT}
    !git clone {REPO_URL} project
    %cd {PROJECT_DIR}

!python -m pip install -q -r requirements.txt
print("Project directory:", PROJECT_DIR)

## 4. Verify Project Files

In [ ]:
!ls
!python -m compileall src

## 5. Download CIFAR-10 To Drive

CIFAR-10 is stored under `MyDrive/5329A2/data/`, so it remains available after a Colab runtime reset.

In [ ]:
from torchvision import datasets

datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True)
datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True)
print("CIFAR-10 is ready at:", DATA_DIR)

## 6. Smoke Test

Run a tiny 1-epoch test before the full experiment. This checks data loading, model construction, FLOPs logging, gradient norm logging, and output writing.

In [ ]:
SMOKE_OUTPUT = RESULTS_DIR / 'smoke_test'

!python -m src.train \
  --dataset cifar10 \
  --model-family plain \
  --width-schedule stagewise \
  --depth 2 \
  --width 32 \
  --epochs 1 \
  --batch-size 128 \
  --seed 0 \
  --data-dir "{DATA_DIR}" \
  --output-dir "{SMOKE_OUTPUT}" \
  --measure-flops \
  --track-grad-norm \
  --train-limit 1024 \
  --test-limit 512 \
  --no-progress

## 7. Experiment Settings

Start with 30 epochs to get usable preliminary results. For the final paper, increase to 50 or 100 epochs if time allows.

In [ ]:
TARGET_PARAMS = 250_000
DEPTHS = "2,4,6,8,10,12"
SEEDS = "0,1,2"
EPOCHS = 30
BATCH_SIZE = 128

PLAIN_OUTPUT = RESULTS_DIR / f'cifar10_plain_budget_{TARGET_PARAMS}_e{EPOCHS}'
RESIDUAL_OUTPUT = RESULTS_DIR / f'cifar10_residual_budget_{TARGET_PARAMS}_e{EPOCHS}'

print("Plain output:", PLAIN_OUTPUT)
print("Residual output:", RESIDUAL_OUTPUT)

## 8. Run CIFAR-10 Plain CNN Sweep

This is the main baseline experiment: fixed parameter budget, different depths, automatically matched widths.

In [ ]:
%%time
!python -m src.sweep \
  --dataset cifar10 \
  --model-family plain \
  --width-schedule stagewise \
  --target-params {TARGET_PARAMS} \
  --depths {DEPTHS} \
  --epochs {EPOCHS} \
  --seeds {SEEDS} \
  --batch-size {BATCH_SIZE} \
  --data-dir "{DATA_DIR}" \
  --output-dir "{PLAIN_OUTPUT}" \
  --measure-flops \
  --track-grad-norm \
  --no-progress

## 9. Run CIFAR-10 Residual CNN Sweep

This comparison helps separate depth-width effects from optimization difficulty in deeper plain CNNs.

In [ ]:
%%time
!python -m src.sweep \
  --dataset cifar10 \
  --model-family residual \
  --width-schedule stagewise \
  --target-params {TARGET_PARAMS} \
  --depths {DEPTHS} \
  --epochs {EPOCHS} \
  --seeds {SEEDS} \
  --batch-size {BATCH_SIZE} \
  --data-dir "{DATA_DIR}" \
  --output-dir "{RESIDUAL_OUTPUT}" \
  --measure-flops \
  --track-grad-norm \
  --no-progress

## 10. Generate Figures And Tables

In [ ]:
for output_dir in [PLAIN_OUTPUT, RESIDUAL_OUTPUT]:
    print("Analyzing", output_dir)
    !python -m src.plot_results \
      --results-dir "{output_dir}" \
      --summary "{output_dir / 'sweep_summary.csv'}" \
      --output-dir "{output_dir / 'figures'}"
    !python -m src.analysis \
      --summary "{output_dir / 'sweep_summary.csv'}" \
      --output-dir "{output_dir / 'tables'}"

## 11. Inspect Results

In [ ]:
import pandas as pd
from IPython.display import display, Image

plain_summary = pd.read_csv(PLAIN_OUTPUT / 'sweep_summary.csv')
display(plain_summary.sort_values('best_test_accuracy', ascending=False).head())

plain_fig = PLAIN_OUTPUT / 'figures' / 'accuracy_vs_ratio.png'
if plain_fig.exists():
    display(Image(filename=str(plain_fig)))

residual_summary_path = RESIDUAL_OUTPUT / 'sweep_summary.csv'
if residual_summary_path.exists():
    residual_summary = pd.read_csv(residual_summary_path)
    display(residual_summary.sort_values('best_test_accuracy', ascending=False).head())
    residual_fig = RESIDUAL_OUTPUT / 'figures' / 'accuracy_vs_ratio.png'
    if residual_fig.exists():
        display(Image(filename=str(residual_fig)))

## 12. Optional CIFAR-100 Experiment

Only run this if CIFAR-10 experiments finish and there is enough GPU time. CIFAR-100 helps support claims about task difficulty.

In [ ]:
RUN_CIFAR100 = False

if RUN_CIFAR100:
    datasets.CIFAR100(root=str(DATA_DIR), train=True, download=True)
    datasets.CIFAR100(root=str(DATA_DIR), train=False, download=True)
    CIFAR100_OUTPUT = RESULTS_DIR / f'cifar100_plain_budget_{TARGET_PARAMS}_e{EPOCHS}'
    !python -m src.sweep \
      --dataset cifar100 \
      --model-family plain \
      --width-schedule stagewise \
      --target-params {TARGET_PARAMS} \
      --depths {DEPTHS} \
      --epochs {EPOCHS} \
      --seeds {SEEDS} \
      --batch-size {BATCH_SIZE} \
      --data-dir "{DATA_DIR}" \
      --output-dir "{CIFAR100_OUTPUT}" \
      --measure-flops \
      --track-grad-norm \
      --no-progress

## 13. Zip Results For Download Or Backup

In [ ]:
ARCHIVE_PATH = DRIVE_ROOT / 'cnn_depth_width_results.zip'
!cd "{RESULTS_DIR}" && zip -qr "{ARCHIVE_PATH}" .
print("Archive written to:", ARCHIVE_PATH)

## Notes For The Paper

Use these outputs in the report:

- `sweep_summary.csv`: main numeric results.
- `figures/accuracy_vs_ratio.png`: depth-width ratio vs accuracy.
- `figures/generalization_gap_vs_ratio.png`: overfitting/generalization behavior.
- `figures/accuracy_vs_flops.png`: accuracy vs compute cost.
- `tables/model_ranking.md`: paper-ready ranking table.

Avoid claiming a universal optimal ratio unless it holds across model family, budget, seed, and dataset.